In [ ]:
# Import Data and Setup
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation
!pip install pyLDAvis
import pyLDAvis
import numpy as np
import warnings
warnings.simplefilter("ignore", DeprecationWarning) # This code is extremely spammy otherwise

file_path = "/content/drive/MyDrive/clash_royale_reviews.csv"

df = pd.read_csv(
    file_path,
    usecols=["review_id", "content", "score", "at", "likes", "appVersion"],
    parse_dates=["at"]
)

In [ ]:
# Categorize Reviews Based On Score
conditions = [
(df['score'] > 2),
(df['score'] < 3)
]
choices = [
'positive',
'negative'
]
df['review_category'] = np.select(conditions, choices, default = '')

In [ ]:
#sample smaller portions of reviews to avoid crashing during analysis
sampledf = df.sample(n= 500000, random_state = 123)
smaller_sampledf = df.sample(n = 100000, random_state = 123)
vectorizer = CountVectorizer(min_df=0.005, max_df=0.9, stop_words='english')
clash_dtm = vectorizer.fit_transform(sampledf['content'].values.astype('U'))

In [ ]:
lda = LatentDirichletAllocation(n_components=20, random_state=123)
lda.fit(clash_dtm)

LatentDirichletAllocation(n_components=20, random_state=123)

In [ ]:
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_):
    top_features_ind = topic.argsort()[:-10 - 1:-1]
    top_features = [feature_names[i] for i in top_features_ind]
    print(f"Topic #{topic_idx + 1}: {', '.join(top_features)}")

Topic #1: supercell, cool, game, gameplay, know, enjoy, don, graphics, say, guys
Topic #2: fun, game, worst, interesting, trash, lot, play, used, don, make
Topic #3: like, really, game, cards, updates, making, don, lot, make, new
Topic #4: time, chest, open, chests, long, better, way, takes, stars, lot
Topic #5: win, level, game, pay, cards, players, match, higher, matchmaking, battle
Topic #6: amazing, game, lvl, friends, pretty, 10, connection, lose, easy, say
Topic #7: play, game, graphics, time, pass, good, free, easy, day, want
Topic #8: bad, hard, game, work, dont, boring, thing, gets, little, free
Topic #9: just, playing, game, app, years, awsome, stop, ve, day, enjoy
Topic #10: game, nice, addictive, enjoy, lot, play, easy, stars, graphics, battle
Topic #11: super, game, thanks, star, royal, guys, needs, clash, make, lot
Topic #12: update, new, game, money, loved, cards, want, spend, im, don
Topic #13: best, game, played, world, games, wow, mobile, ve, phone, years
Topic #14: g

In [ ]:
doctopic = lda.transform(clash_dtm)

In [ ]:
vis_data = pyLDAvis.prepare(topic_term_dists = lda.components_, # topic-word matrix
                            doc_topic_dists = doctopic, # document-topic matrix
                            doc_lengths = np.array(clash_dtm.sum(axis = 1)).flatten(), # The package expects flat arrays, so we need to convert to numpy array first and then flatten
                            vocab = vectorizer.get_feature_names_out(),
                            term_frequency = np.array(clash_dtm.sum(axis = 0)).flatten(),
                            sort_topics = True) # Set this to True to sort topics by size

In [ ]:
pyLDAvis.display(vis_data)

In [46]:
#topic models for only positive reviews
positive_df = df[df['review_category'] == 'positive']
pos_sample = positive_df.sample(n=250000, random_state=123)
vectorizer = CountVectorizer(min_df=50, max_df=0.9, stop_words='english')
positive_clash_dtm = vectorizer.fit_transform(pos_sample['content'].values.astype('U'))

In [47]:
pos_lda = LatentDirichletAllocation(n_components=20, random_state=123)
pos_lda.fit(positive_clash_dtm)

LatentDirichletAllocation(n_components=20, random_state=123)

In [48]:
pos_feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(pos_lda.components_):
    top_features_ind = topic.argsort()[:-10 - 1:-1]
    top_features = [pos_feature_names[i] for i in top_features_ind]
    print(f"Topic #{topic_idx + 1}: {', '.join(top_features)}")

Topic #1: supercell, game, games, thanks, recommend, guys, wonderful, job, work, lots
Topic #2: time, clan, killer, addicted, game, real, war, takes, great, long
Topic #3: graphics, game, chests, open, gameplay, long, wait, needs, liked, little
Topic #4: game, great, good, strategy, overall, far, yeah, enjoying, graphics, fav
Topic #5: chest, make, legendary, cards, card, add, gems, plz, stars, want
Topic #6: wow, lose, lol, trophies, game, mind, hate, juego, es, que
Topic #7: game, think, time, play, pass, day, enjoy, fast, way, favorite
Topic #8: nice, awesome, game, awsome, bro, addiction, supper, damn, muito, bom
Topic #9: good, loved, app, pretty, hog, games, work, job, rider, update
Topic #10: cards, level, game, people, arena, 10, players, fantastic, higher, lvl
Topic #11: fun, play, game, addicting, friends, easy, card, legendary, excellent, hard
Topic #12: love, game, download, funny, challenging, absolutely, luv, fine, play, poop
Topic #13: addictive, game, win, pay, new, upd

In [49]:
pos_doctopic = pos_lda.transform(positive_clash_dtm)

In [50]:
pos_vis_data = pyLDAvis.prepare(topic_term_dists = pos_lda.components_, # topic-word matrix
                            doc_topic_dists = pos_doctopic, # document-topic matrix
                            doc_lengths = np.array(positive_clash_dtm.sum(axis = 1)).flatten(), # The package expects flat arrays, so we need to convert to numpy array first and then flatten
                            vocab = vectorizer.get_feature_names_out(),
                            term_frequency = np.array(positive_clash_dtm.sum(axis = 0)).flatten(),
                            sort_topics = True) # Set this to True to sort topics by size

In [51]:
pyLDAvis.display(pos_vis_data)

In [52]:
#topic models for only negative reviews
negative_df = df[df['review_category'] == 'negative']
neg_sample = negative_df.sample(n=250000, random_state=123)
vectorizer = CountVectorizer(min_df=50, max_df=0.9, stop_words='english')
negative_clash_dtm = vectorizer.fit_transform(neg_sample['content'].values.astype('U'))

In [53]:
neg_lda = LatentDirichletAllocation(n_components=20, random_state=123)
neg_lda.fit(negative_clash_dtm)

LatentDirichletAllocation(n_components=20, random_state=123)

In [54]:
neg_feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(neg_lda.components_):
    top_features_ind = topic.argsort()[:-10 - 1:-1]
    top_features = [neg_feature_names[i] for i in top_features_ind]
    print(f"Topic #{topic_idx + 1}: {', '.join(top_features)}")

Topic #1: game, fix, connection, problem, lost, internet, lag, wifi, network, battle
Topic #2: game, hate, hard, star, stars, nice, games, coc, better, like
Topic #3: arena, cards, legendary, card, got, people, game, like, playing, just
Topic #4: game, good, fun, playing, years, just, used, boring, really, ve
Topic #5: update, game, play, new, app, 50, fix, open, screen, load
Topic #6: don, game, sucks, like, know, play, want, horrible, just, guys
Topic #7: game, players, play, player, fair, people, recommend, ok, playing, don
Topic #8: play, game, mega, knight, nerf, just, free, like, dont, evo
Topic #9: clash, royale, game, pass, new, clan, update, clans, supercell, players
Topic #10: win, pay, game, just, fun, games, garbage, cards, extremely, card
Topic #11: level, bad, cards, higher, matchmaking, game, 10, 13, 11, 12
Topic #12: lvl, p2w, que, el, juego, es, la, se, en, vs
Topic #13: game, worst, trash, stupid, bad, played, world, matchmaking, seen, unbalanced
Topic #14: chest, che

In [55]:
neg_doctopic = neg_lda.transform(negative_clash_dtm)

In [56]:
neg_vis_data = pyLDAvis.prepare(topic_term_dists = neg_lda.components_, # topic-word matrix
                            doc_topic_dists = neg_doctopic, # document-topic matrix
                            doc_lengths = np.array(negative_clash_dtm.sum(axis = 1)).flatten(), # The package expects flat arrays, so we need to convert to numpy array first and then flatten
                            vocab = vectorizer.get_feature_names_out(),
                            term_frequency = np.array(negative_clash_dtm.sum(axis = 0)).flatten(),
                            sort_topics = True) # Set this to True to sort topics by size

In [57]:
pyLDAvis.display(neg_vis_data)